In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
import h5py

import openquantum_sde
from openquantum_sde.io import load_trajectory, load_params


In [ ]:
# Lookup data folder
if "DATA" in os.environ:
    base_dir = Path(os.environ["DATA"]).expanduser()
else:
    base_dir = Path(".")  # current folder
PROJECT_NAME = "openquantum_sde"
output_dir = base_dir / PROJECT_NAME / 'figs'

In [ ]:
# Load data
epsilon_array = [11.0, 15.0, 20.0]
time_array = [None] * len(epsilon_array)
current_array = [None] * len(epsilon_array)
max_time_iteration = 12500 #25000 #2500 = 100
offsets = [10000, 3750, 1350]
filenum = [1, 1, 1]
for i, epsilon in enumerate(epsilon_array):
    SIM_NAME = "transmon_cavity_eps_" + str(epsilon) + "/data"
    simulation_dir = base_dir / PROJECT_NAME / SIM_NAME
    if not os.path.exists(simulation_dir):
        SIM_NAME = "transmon_cavity_eps_" + str(int(epsilon)) + "/data"
        simulation_dir = base_dir / PROJECT_NAME / SIM_NAME

    filename = Path(simulation_dir) / f"traj_CK_{filenum[i]:04d}.h5"
    offset = offsets[i]

    with h5py.File(filename, "r") as f:
        time_array[i] = f["time"][0:max_time_iteration]
        current_array[i] = f["traj_current"][offset:125000+offset]

In [ ]:
# Minimas in phase space depending on drive epsilon (for phase space plots)
minimas_by_epsilon = {
    11: [0.00 + 0.00j, 2.18 + 4.39j, 9.79 + 3.45j],
    12: [0.00 + 0.00j, 2.15 + 4.60j, 9.84 + 4.61j],
    13: [0.00 + 0.01j, 2.14 + 4.82j, 9.85 + 5.57j],
    14: [0.00 + 0.01j, 2.14 + 5.04j, 9.86 + 6.39j],
    15: [0.00 + 0.01j, 2.15 + 5.25j, 9.85 + 7.12j],
    16: [0.00 + 0.01j, 2.17 + 5.47j, 9.86 + 7.78j],
    17: [0.00 + 0.01j, 2.19 + 5.70j, 9.89 + 8.39j],
    18: [0.00 + 0.01j, 2.23 + 5.93j, 9.93 + 8.95j],
    19: [0.00 + 0.01j, 2.27 + 6.16j, 9.98 + 9.49j],
    20: [0.00 + 0.01j, 2.32 + 6.40j, 10.04 + 10.00j],
}

In [ ]:
# Set up triplot grid, top row current, bottom phase space
fig = plt.figure(figsize=(12, 7))
savefig = True

# Set global font to sans-serif (Helvetica/Arial)
#plt.rcParams['font.family'] = 'sans-serif'
#plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams.update({
    "font.family": "STIXGeneral",
    "mathtext.fontset": "stix",
    "font.size": 14,
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 14,
})

gs = fig.add_gridspec(
    2, 3,
    height_ratios=[1, 2],   # top row = 1/3 of bottom row
    hspace=0.2,
    wspace=0.3
)

# Top row: time series
ax_time = [fig.add_subplot(gs[0, i]) for i in range(3)]

# Bottom row: phase spaces
#ax_phase = [fig.add_subplot(gs[1, i], aspect='equal') for i in range(3)]
ax_phase = [fig.add_subplot(gs[1, i]) for i in range(3)]

colors = [
    '#355C7D',  # blue
    '#4C956C',  # muted green
    '#E17C05',  # orange
    '#7A5195',  # purple
]

# Plot time series for current
for i in range(3):
    times = time_array[i][0:max_time_iteration]
    traj_current = current_array[i][0:max_time_iteration]
    epsilon = epsilon_array[i]
    ax_time[i].plot(times, traj_current.real, label=r'$\mathrm{Re}(\alpha)$', lw=0.7, color=colors[1])
    ax_time[i].plot(times, traj_current.imag, label=r'$\mathrm{Im}(\alpha)$', lw=0.7,  color=colors[2])
    ax_time[i].plot(times, (traj_current*traj_current.conjugate()).real, label=r'$|\alpha^2|$', lw=0.7,  color=colors[0])
    ax_time[i].set_title(r'$\epsilon=$'+str(epsilon))

    traj_current = current_array[i]
    ax_phase[i].plot(traj_current.real, traj_current.imag, lw=0.2, color='k', label=r'Trajectory') # ($t=5000$)')
    minimas = np.array(minimas_by_epsilon.get(int(epsilon), []))
    ax_phase[i].plot(minimas.real, minimas.imag, 'rx', label='Minimas')

ylim_current = [-10,150]

for ax in ax_time:
    ax.set_xlabel('Time (t)')
    ax.set_ylim(ylim_current)
    ax.grid(True, alpha=0.3)
    leg = ax.legend(loc='upper left', 
              framealpha=1, 
              bbox_to_anchor=(0.7, 1.1), 
              fancybox=True,
              edgecolor='0.5',
              borderpad=0.3,
              handlelength=1.2,
              handletextpad=0.4,
              labelspacing=0.3)
    
    for line in leg.get_lines():
        line.set_linewidth(2.0)

    # Set Background Colors
    ax.set_facecolor('gainsboro')  # Inner plot area background (gainsboro, lightgray, whitesmoke)
    # Configure White Grid Lines
    ax.grid(True, color='white', linestyle='-', linewidth=1.2)
    # Improve visibility of ticks and labels against light gray
    ax.tick_params(colors='black')
    for spine in ax.spines.values():
        spine.set_edgecolor('black')


xlim_phase = [-2.0, 11]
ylim_phase = [-2.0, 11]

for ax in ax_phase:
    ax.set_xlabel(r'$\mathrm{Re}(\alpha)$')
    ax.set_ylabel(r'$\mathrm{Im}(\alpha)$')
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(xlim_phase)
    ax.set_ylim(ylim_phase)
    ax.grid(True, alpha=0.3)
    leg2 = ax.legend(loc='upper left', 
              framealpha=1, 
              fancybox=True,
              edgecolor='0.5',
              borderpad=0.3,
              handlelength=1.2,
              handletextpad=0.4,
              labelspacing=0.3) # bbox_to_anchor=(-0.1, 1.01))

    for line in leg2.get_lines():
        line.set_linewidth(2.0)
    

    # Set Background Colors
    ax.set_facecolor('gainsboro')         # Inner plot area background
    # Configure White Grid Lines
    ax.grid(True, color='white', linestyle='-', linewidth=1.2)
    # Improve visibility of ticks and labels against light gray
    ax.tick_params(colors='black')
    for spine in ax.spines.values():
        spine.set_edgecolor('black')


fig.subplots_adjust(
    left=0.045,
    right=0.995,
    bottom=0.06,
    top=0.94,
    wspace=0.12,
    hspace=0.12
)

if savefig:
    fig.savefig(
        output_dir / 'triplot_current.png',
        dpi=300,
        bbox_inches='tight'
        )